# Detección de fatiga en la conducción de vehículos

## Información
Auctorvolumus nostrum iaculis mus torquent ac atqui vehicula scelerisque at nonumy.  Luctusaliquet ponderum nisl mea scelerisque nec graeci.
Torquentac in dicant scripserit oratio regione comprehensam nonumy auctor aliquid conclusionemque delicata periculis fames quem sale iusto euripidis.  Erremvolutpat tempus sed lorem habitasse legere conceptam oporteat similique facilisis.  Nonposse usu erat ea salutatus suspendisse.  Aliquetbrute doctus fastidii moderatius ignota vero mus.  Utamurpurus lacinia ex antiopam ne deserunt.  Comprehensammetus voluptatum praesent egestas consul.


## Objetivo
Auctorvolumus nostrum iaculis mus torquent ac atqui vehicula scelerisque at nonumy.  Luctusaliquet ponderum nisl mea scelerisque nec graeci.
Torquentac in dicant scripserit oratio regione comprehensam nonumy auctor aliquid conclusionemque delicata periculis fames quem sale iusto euripidis.  Erremvolutpat tempus sed lorem habitasse legere conceptam oporteat similique facilisis.  Nonposse usu erat ea salutatus suspendisse.  Aliquetbrute doctus fastidii moderatius ignota vero mus.  Utamurpurus lacinia ex antiopam ne deserunt.  Comprehensammetus voluptatum praesent egestas consul.

In [ ]:
UNSPLITTED_FOLDER_IMAGES = './datasets/dmd/labels'

Cargamos algunas imágenes para comprobar la naturaleza de las mismas.

In [ ]:
#!pip install Pillow
import os
import tensorflow as tf
import PIL
from tensorflow.keras.preprocessing import image

tf.keras.backend.clear_session()
image.load_img(UNSPLITTED_FOLDER_IMAGES + '/eyes_state_open/0.jpg')

## Análisis exploratorio de datos
Antes de construir el modelo, es fundamental llevar a cabo un proceso de exploración de los datos para identificar y resolver cualquier posible problema. Durante este análisis, verificaremos el equilibrio entre las diferentes clases para evitar cualquier sesgo que pueda afectar a nuestro modelo. Además, nos aseguraremos de que todas las imágenes tengan tres canales y, en caso necesario, normalizaremos los datos.

## Dataset de train, test y validación
Para crear los dataset de train y test, vamos a utilizar la función splitfolders. Como ya hemos descargado algunas fotos de internet para la validación, únicamente dividiremos nuestro dataset original en train y test con una proporción de 80/20.

In [ ]:
INPUT_SHAPE = (224, 224, 3)
DATA_DIR = './datasets/dmd/labels_split'
BATCH_SIZE = 32
IMG_SIZE = (180, 180)
VAL_SPLIT = 0.2

import splitfolders
if not os.path.exists('./datasets/dmd/labels_split'):
    splitfolders.ratio('./datasets/dmd/labels', './datasets/dmd/labels_split', seed=1, ratio=(.9, 0, .1))

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='training',
    color_mode='rgb',
    seed=1,
    shuffle=0,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='validation',
    color_mode='rgb',
    seed=1,
    shuffle=0,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE)

## Clases equilibradas
Para empezar, verificamos la distribución equilibrada de las clases, lo cual implica que todas ellas contengan la misma cantidad de imágenes. Para llevar a cabo esta comprobación, hemos implementado la función "check_classes", que realiza el análisis y nos proporciona un mensaje indicando si las clases están balanceadas o no, junto con el tamaño de cada una de ellas. Como entrada, solo necesitamos especificar la ruta principal y las diferentes carpetas que contienen las imágenes correspondientes a cada clase:

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)

def check_classes(base_path, folders):
    number_files = []
    for c in folders:
        dir = base_path + '/' + c
        list = os.listdir(dir)
        number_files.append(len(list))
    if len(set(number_files)) == 1: # convertimos la lista en set, que elimina los valores duplicados.
    # si su longitud es 1 significa que todos los valores se repiten
        return 'Clases equilibradas:'+ str(number_files)
    else:
       return "!!!Clases desequilibradas:" + str(number_files)

print(check_classes(DATA_DIR, class_names))

Según podemos observar en la salida previa, las clases presentan una distribución equilibrada, contando todas ellas con un total de 81 imágenes. No obstante, es importante resaltar que en realidad cada clase debería contener 47.567 imágenes.

## Número de canales
Además, es crucial verificar que todas las imágenes tengan tres canales para evitar posibles inconvenientes más adelante. Para facilitar esta comprobación, hemos implementado la función "check_channels". Esta función requiere como entrada la ruta principal, los nombres de las subcarpetas y el número de canales que deseamos que tengan las imágenes, en este caso, 3.

## Normalización
Por último, en cuanto a la normalización, no sería necesario llevarla a cabo ya que las redes que vamos a utilizar ya contienen una función de preprocesado que importamos directamente desde keras y que realiza todo el tratamiento de los datos para adaptar el input a la red neuronal en cuestión.

## VGG19
Comenzamos la modelización con la red VGG19 y añadimos las capas necesarias.

In [ ]:
from keras.layers import Dense,GlobalAveragePooling2D
from keras.applications import VGG19

base_model=VGG19(weights='imagenet', include_top=False)

vgg = base_model.output
vgg = GlobalAveragePooling2D()(vgg)
vgg = Dense(512, activation='relu')(vgg)
outputs = Dense(num_classes, activation='softmax')(vgg)

Creamos el modelo y obtenemos el summary

In [13]:
# conda install -c conda-forge python-graphviz
#!pip install keras_visualizer
#!pip install pip install git+https://github.com/raghakot/keras-vis.git -U
from keras.models import Model
from keras.utils.vis_utils import plot_model
from keras_visualizer import visualizer

model = Model(inputs=base_model.input, outputs=outputs)
model.summary()
# visualizer(model,file_name='modelVisualization.png', file_format='png', view=True)

Model: "model_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, None, None, 3)]   0         
                                                                 
 block1_conv1 (Conv2D)       (None, None, None, 64)    1792      
                                                                 
 block1_conv2 (Conv2D)       (None, None, None, 64)    36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, None, None, 64)    0         
                                                                 
 block2_conv1 (Conv2D)       (None, None, None, 128)   73856     
                                                                 
 block2_conv2 (Conv2D)       (None, None, None, 128)   147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, None, None, 128)   0   

In [14]:
print("Modelo base:", len(base_model.layers), "\nModelo:", len(model.layers))

Modelo base: 22 
Modelo: 25


Indicamos a partir de qué capa el modelo empieza a entrenar. Esto es importante porque si lo entrenamos de 0
nos va a llevar mucho más tiempo. Así , el model sólo aprenderá de las capas que le indiquemos.

In [15]:
for layer in model.layers[:19]:
    layer.trainable=False
for layer in model.layers[19:]:
    layer.trainable=True
model.summary()

Model: "model_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, None, None, 3)]   0         
                                                                 
 block1_conv1 (Conv2D)       (None, None, None, 64)    1792      
                                                                 
 block1_conv2 (Conv2D)       (None, None, None, 64)    36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, None, None, 64)    0         
                                                                 
 block2_conv1 (Conv2D)       (None, None, None, 128)   73856     
                                                                 
 block2_conv2 (Conv2D)       (None, None, None, 128)   147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, None, None, 128)   0   

Vemos como el número de parámetros entrenables es de 20.288.579, correspondientes a las capas que no hemos congelado y que vamos a entrenar a continuación. Para ello, creamos el generador de train y test usando como función de preproceso la de la red VGG19, fijamos los directorios donde se encuentran los dataset e indicamos como tamaño objetivo 224x224:

In [16]:
from keras.preprocessing.image import ImageDataGenerator
from keras.applications.vgg19 import preprocess_input as preprocess_vgg19

# Cremos el generador especificando que queremos que use como función de preproceso la de VGG19
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_vgg19) # lo especificamos aquí
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_vgg19)
# Podríamos usar el mismo en este caso para train y test pero así queda más claro

train_path = './datasets/dmd/labels_split/train'
test_path = './datasets/dmd/labels_split/test'
# Generador para train
train_generator = train_datagen.flow_from_directory(train_path, # this is where you specify the path to the main data folder
                                                    target_size=(224,224),
                                                    # default parameters
                                                    color_mode='rgb',
                                                    batch_size=2,
                                                    class_mode='categorical',
                                                    shuffle=False)

# Generador para test
validation_generator = test_datagen.flow_from_directory(
                        test_path,
                        target_size=(224,224),
                        color_mode='rgb',
                        batch_size=2,
                        class_mode='categorical',
                        shuffle=False)

Found 23779 images belonging to 7 classes.
Found 2647 images belonging to 7 classes.


Compilamos el modelo utilizando como optimizador "Adam" y como función de coste la entropía cruzada (Cross-entropy). Lo entrenaremos para 50 épocas pero activamos los callbacks de early_stopping:

In [18]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)

model.compile(optimizer='Adam',loss='categorical_crossentropy', metrics=['accuracy'])
step_size_train=train_generator.n//train_generator.batch_size
modelvgg19 = model.fit_generator(
    generator=train_generator,
    validation_data = validation_generator,
    steps_per_epoch=step_size_train,
    epochs=50,
    callbacks=[early_stop])

C:\Users\david\AppData\Local\Temp\ipykernel_18220\2991857933.py:5: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  modelvgg19 = model.fit_generator(


NameError: name 'scipy' is not defined